# 🔍 Hybrid RAG Pipeline
**Architecture:** BM25 + FAISS → Cross-Encoder Reranker → Parent-Document → OpenRouter LLM

### Before you start:
1. Choose `Runtime → Change runtime type → T4 GPU` in the menu
2. Fill the **CONFIG** section with your API keys
3. Run all cells in order

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q faiss-cpu rank_bm25 sentence-transformers openai tiktoken langchain langchain-community
print('✅ Dependencies installed')

In [ ]:
# @title
# ============================================================
# CELL 2: CONFIG — fill in your values
# ============================================================
import os

# --- OpenRouter ---
OPENROUTER_API_KEY = "sk-"   # https://openrouter.ai/keys
OPENROUTER_MODEL   = ""  # or any other model from OpenRouter

# --- Embedding model (runs locally, free) ---
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"


# --- Reranker (runs locally, ~80MB) ---
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# --- Retrieval settings ---
CHILD_CHUNK_SIZE   = 200   # tokens — small chunk size for precise search
PARENT_CHUNK_SIZE  = 800   # tokens — large chunk size to give context to the LLM
CHILD_OVERLAP      = 20    # overlap tokens between child chunks
TOP_K_RETRIEVAL    = 10    # number of chunks to fetch from BM25 and FAISS individually
TOP_N_RERANK       = 4     # number of chunks to keep after reranking

# --- Google Drive ---
USE_DRIVE          = True
DRIVE_SAVE_DIR     = "/content/drive/MyDrive/rag_index"  # folder on your Google Drive

print('✅ Config loaded')
print(f'   Generation model : {OPENROUTER_MODEL}')
print(f'   Embedding model  : {EMBED_MODEL_NAME}')
print(f'   Reranker model   : {RERANKER_MODEL_NAME}')

In [ ]:
# ============================================================
# CELL 3: Imports and Google Drive mounting
# ============================================================
import json, pickle, os, re, time
import numpy as np
import faiss
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

# --- OpenRouter client (uses OpenAI SDK — OpenRouter is compatible) ---
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# --- Mount Google Drive ---
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    print(f'✅ Drive mounted → {DRIVE_SAVE_DIR}')

print('✅ Imports loaded')
print(f'   GPU is available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU model: {torch.cuda.get_device_name(0)}')
    print(f'   Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 4: Data structures
# ============================================================
@dataclass
class Chunk:
    """One document chunk"""
    chunk_id    : str
    text        : str
    parent_id   : str          # link to the parent chunk
    source      : str = ""     # filename or URL
    metadata    : Dict = field(default_factory=dict)

@dataclass
class RetrievedChunk:
    """Search result — child chunk + its parent text + score"""
    chunk      : Chunk
    parent_text: str
    score      : float
    method     : str   # 'bm25', 'faiss', 'rrf'

print('✅ Dataclasses defined')

In [ ]:
# ============================================================
# CELL 5: Chunking functions (child + parent)
# ============================================================
import tiktoken

TOKENIZER = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(TOKENIZER.encode(text))

def split_into_chunks(text: str, max_tokens: int, overlap: int = 0) -> List[str]:
    """Splits text into chunks of max_tokens with overlap."""
    tokens = TOKENIZER.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(TOKENIZER.decode(chunk_tokens))
        if end == len(tokens):
            break
        start += max_tokens - overlap
    return chunks

def build_parent_child_chunks(
    documents: List[Dict],   # [{"text": str, "source": str}, ...]
    child_size: int  = CHILD_CHUNK_SIZE,
    parent_size: int = PARENT_CHUNK_SIZE,
    child_overlap: int = CHILD_OVERLAP
) -> Tuple[List[Chunk], Dict[str, str]]:
    """
    Returns:
      child_chunks  — list of small chunks for search
      parent_store  — dictionary {parent_id: parent_text} to feed into the LLM
    """
    child_chunks  = []
    parent_store  = {}

    for doc_idx, doc in enumerate(documents):
        source = doc.get("source", f"doc_{doc_idx}")
        # 1. Cut text into large parent chunks
        parents = split_into_chunks(doc["text"], parent_size)
        for p_idx, parent_text in enumerate(parents):
            parent_id = f"{source}::p{p_idx}"
            parent_store[parent_id] = parent_text
            # 2. Cut parent text into small child chunks inside that parent
            children = split_into_chunks(parent_text, child_size, child_overlap)
            for c_idx, child_text in enumerate(children):
                child_id = f"{parent_id}::c{c_idx}"
                child_chunks.append(Chunk(
                    chunk_id  = child_id,
                    text      = child_text,
                    parent_id = parent_id,
                    source    = source
                ))

    print(f'✅ Chunking finished: {len(parent_store)} parent chunks, {len(child_chunks)} child chunks')
    return child_chunks, parent_store

In [ ]:
# ============================================================
# CELL 6: Load models (Embedding + Reranker)
# ============================================================
print('⏳ Loading embedding model...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Embedding model loaded')

print('⏳ Loading cross-encoder reranker...')
reranker = CrossEncoder(RERANKER_MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Reranker loaded')

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'   VRAM used: {used:.1f} / {total:.1f} GB')

In [ ]:
# ============================================================
# CELL 7: Build BM25 + FAISS indexes
# ============================================================
import re

def clean_tokenize(text: str) -> List[str]:
    """Lowercase and strip punctuation to ensure clean keyword matching."""
    # Removes punctuation like ?, ., ,, !, but keeps hyphens inside words
    cleaned = re.sub(r'[^\w\s-]', '', text.lower())
    return cleaned.split()

def build_bm25_index(child_chunks: List[Chunk]) -> BM25Okapi:
    """Tokenize words and build the BM25 index."""
    tokenized = [chunk.text.lower().split() for chunk in child_chunks]
    return BM25Okapi(tokenized)

def build_faiss_index(child_chunks: List[Chunk], batch_size: int = 64) -> Tuple[faiss.Index, np.ndarray]:
    """Encode all child chunks and build the FAISS vector index."""
    texts = [chunk.text for chunk in child_chunks]
    print(f'⏳ Encoding {len(texts)} chunks in batches of {batch_size}...')
    embeddings = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True  # normalize for cosine similarity
    )
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product = Cosine Similarity on normalized vectors

    # SAFE GPU CHECK: Only use FAISS GPU resources if both CUDA is active and FAISS has GPU support
    if torch.cuda.is_available() and hasattr(faiss, 'StandardGpuResources'):
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)

    index.add(embeddings)
    print(f'✅ FAISS index built: {index.ntotal} vectors, dimensions={dim}')
    return index, embeddings

def save_indices(child_chunks, parent_store, bm25_index, embeddings, save_dir=DRIVE_SAVE_DIR):
    """Save all indexes to Google Drive."""
    # Get FAISS CPU index to save it correctly
    cpu_index = faiss.index_gpu_to_cpu(faiss_index) if hasattr(faiss_index, 'getDevice') else faiss_index
    faiss.write_index(cpu_index, f"{save_dir}/faiss.index")
    # Save the rest of the data using pickle
    with open(f"{save_dir}/data.pkl", 'wb') as f:
        pickle.dump({
            'child_chunks' : child_chunks,
            'parent_store' : parent_store,
            'bm25_index'   : bm25_index,
            'embeddings'   : embeddings
        }, f)
    print(f'✅ Indexes saved → {save_dir}')

def load_indices(save_dir=DRIVE_SAVE_DIR):
    """Load indexes from Drive to avoid recalculating them."""
    with open(f"{save_dir}/data.pkl", 'rb') as f:
        data = pickle.load(f)
    cpu_index = faiss.read_index(f"{save_dir}/faiss.index")

    # SAFE GPU CHECK: Only use FAISS GPU resources if both CUDA is active and FAISS has GPU support
    if torch.cuda.is_available() and hasattr(faiss, 'StandardGpuResources'):
        res       = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        data['faiss_index'] = gpu_index
    else:
        data['faiss_index'] = cpu_index

    print(f'✅ Indexes loaded from Drive ({data["faiss_index"].ntotal} vectors)')
    return data


In [ ]:
# ============================================================
# CELL 8: Hybrid Search (BM25 + FAISS + RRF)
# ============================================================
def bm25_search(query: str, child_chunks: List[Chunk], bm25_index: BM25Okapi, top_k: int) -> List[Tuple[int, float]]:
    """Returns [(chunk_idx, score), ...]"""
    tokens = clean_tokenize(query)
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices if scores[i] > 0]

def faiss_search(query: str, faiss_index, top_k: int) -> List[Tuple[int, float]]:
    """Returns [(chunk_idx, score), ...]"""
    q_emb = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, indices = faiss_index.search(q_emb, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx >= 0]

def reciprocal_rank_fusion(
    *ranked_lists: List[Tuple[int, float]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """
    RRF: score(d) = sum( 1 / (k + rank(d)) )
    k=60 is the standard value from the original paper.
    This method blends different search scores by using ranks only.
    """
    rrf_scores: Dict[int, float] = {}
    for ranked_list in ranked_lists:
        for rank, (doc_idx, _) in enumerate(ranked_list):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(
    query       : str,
    child_chunks: List[Chunk],
    parent_store: Dict[str, str],
    bm25_index  : BM25Okapi,
    faiss_index ,
    top_k       : int = TOP_K_RETRIEVAL
) -> List[RetrievedChunk]:
    """BM25 + FAISS → RRF → List of RetrievedChunks"""
    bm25_results  = bm25_search(query, child_chunks, bm25_index, top_k)
    faiss_results = faiss_search(query, faiss_index, top_k)
    merged        = reciprocal_rank_fusion(bm25_results, faiss_results)

    results = []
    for chunk_idx, rrf_score in merged[:top_k]:
        chunk = child_chunks[chunk_idx]
        results.append(RetrievedChunk(
            chunk       = chunk,
            parent_text = parent_store.get(chunk.parent_id, chunk.text),
            score       = rrf_score,
            method      = 'rrf'
        ))
    return results

print('✅ Hybrid search functions defined')

In [ ]:
# ============================================================
# CELL 9: Cross-Encoder Reranker
# ============================================================
def rerank(
    query   : str,
    results : List[RetrievedChunk],
    top_n   : int = TOP_N_RERANK
) -> List[RetrievedChunk]:
    """
    The Cross-encoder evaluates pairs of (query, child_text) and yields a score.
    We rerank based on the child chunk text, but we give the parent chunk text to the LLM.
    """
    if not results:
        return []
    pairs  = [(query, r.chunk.text) for r in results]
    scores = reranker.predict(pairs)
    for r, score in zip(results, scores):
        r.score = float(score)
    reranked = sorted(results, key=lambda x: x.score, reverse=True)
    # Remove duplicates — do not send the exact same parent document twice
    seen_parents, final = set(), []
    for r in reranked:
        if r.chunk.parent_id not in seen_parents:
            seen_parents.add(r.chunk.parent_id)
            final.append(r)
        if len(final) >= top_n:
            break
    return final

print('✅ Reranker function defined')

In [ ]:
# ============================================================
# CELL 10: OpenRouter Answer Generation
# ============================================================
RAG_SYSTEM_PROMPT = """You are a helpful assistant. Answer the question ONLY using the text context given below.
If you cannot find the answer in the context, say that you do not know.
Mention sources if they are specified in the context."""

def build_context_string(reranked_results: List[RetrievedChunk]) -> str:
    """Combine parent chunk texts to build the context prompt for the LLM."""
    parts = []
    for i, r in enumerate(reranked_results, 1):
        source_label = f"[Source: {r.chunk.source}]" if r.chunk.source else ""
        parts.append(f"--- Source Fragment {i} {source_label} ---\n{r.parent_text}")
    return "\n\n".join(parts)

def generate_answer(
    query  : str,
    context: str,
    model  : str = OPENROUTER_MODEL,
    max_tokens: int = 1024
) -> str:
    """Call OpenRouter API using OpenAI-compatible client library."""
    user_message = f"""Context:\n{context}\n\n---\nQuestion: {query}\n\nAnswer:"""
    response = openrouter_client.chat.completions.create(
        model      = model,
        messages   = [
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        max_tokens = max_tokens,
        temperature= 0.1,  # low temperature for precise factual answers
    )
    return response.choices[0].message.content

print('✅ OpenRouter functions defined')

In [ ]:
# ============================================================
# CELL 11: Full RAG Pipeline
# ============================================================
def rag_query(
    query       : str,
    child_chunks: List[Chunk],
    parent_store: Dict[str, str],
    bm25_index  : BM25Okapi,
    faiss_index ,
    top_k       : int = TOP_K_RETRIEVAL,
    top_n       : int = TOP_N_RERANK,
    verbose     : bool = True
) -> Dict:
    """
    Full RAG Pipeline:
    1. Hybrid retrieval (BM25 + FAISS → merge ranks with RRF)
    2. Rerank chunks using Cross-encoder
    3. Expand retrieved child chunks to parent chunks
    4. Ask OpenRouter LLM to write the answer
    """
    t0 = time.time()

    # Stage 1: Hybrid retrieval
    retrieved = hybrid_search(query, child_chunks, parent_store, bm25_index, faiss_index, top_k)
    t1 = time.time()

    # Stage 2: Reranking & Parent chunk expansion
    reranked  = rerank(query, retrieved, top_n)
    t2 = time.time()

    # Stage 3: LLM generation
    context   = build_context_string(reranked)
    answer    = generate_answer(query, context)
    t3 = time.time()

    if verbose:
        print(f"\n{'='*60}")
        print(f"QUERY: {query}")
        print(f"{'='*60}")
        print(f"\n📊 STATISTICS:")
        print(f"   Hybrid retrieval : {len(retrieved)} chunks retrieved in {t1-t0:.2f}s")
        print(f"   After reranking  : {len(reranked)} chunks kept in {t2-t1:.2f}s")
        print(f"  LLM generation   : {t3-t2:.2f}s | model: {OPENROUTER_MODEL}")
        print(f"   Total time       : {t3-t0:.2f}s")
        print(f"\n📎 CHUNKS USED AS CONTEXT:")
        for i, r in enumerate(reranked, 1):
            print(f"   [{i}] rerank_score={r.score:.3f} | source: {r.chunk.source} | text: {r.chunk.text[:80].strip()}...")
        print(f"\n🤖 ANSWER:")
        print(answer)
        print(f"{'='*60}")

    return {
        'answer'   : answer,
        'retrieved': retrieved,
        'reranked' : reranked,
        'context'  : context,
        'timings'  : {'retrieval': t1-t0, 'rerank': t2-t1, 'generation': t3-t2}
    }

print('✅ RAG pipeline defined')

In [ ]:
# ============================================================
# CELL 12: Load your documents
# ============================================================

import glob
import os
documents = []
# - PyMuPDF4LLM: "*__pymupdf4llm.md"
# - Docling: "*__docling.md"
search_pattern = "/content/drive/MyDrive/RAG/parsed_out/cleaned/*__pymupdf4llm.md"
cleaned_files = glob.glob(search_pattern)
for path in sorted(cleaned_files):
    with open(path, 'r', encoding='utf-8') as f:
        text_content = f.read()
        documents.append({
            'text': text_content,
            'source': os.path.basename(path)
        })
print(f'📚 Succesfully loaded num of docs: {len(documents)}')
for d in documents[:5]:
    print(f'   - {d["source"]}: {token_len(d["text"])} tokens')

In [ ]:
# ============================================================
# CELL 13: Build or Load indexes
# ============================================================
INDEX_EXISTS = USE_DRIVE and os.path.exists(f"{DRIVE_SAVE_DIR}/data.pkl")

if INDEX_EXISTS:
    print('📂 Found saved indexes on Google Drive, loading them...')
    data         = load_indices()
    child_chunks = data['child_chunks']
    parent_store = data['parent_store']
    bm25_index   = data['bm25_index']
    embeddings   = data['embeddings']
    faiss_index  = data['faiss_index']
else:
    print('🔨 No saved index found. Building indexes from scratch...')
    child_chunks, parent_store = build_parent_child_chunks(documents)
    bm25_index   = build_bm25_index(child_chunks)
    faiss_index, embeddings = build_faiss_index(child_chunks)
    if USE_DRIVE:
        save_indices(child_chunks, parent_store, bm25_index, embeddings)

print(f'\n✅ System is ready for search!')
print(f'   Child chunks  : {len(child_chunks)}')
print(f'   Parent chunks : {len(parent_store)}')
print(f'   FAISS vectors : {faiss_index.ntotal}')

In [ ]:
# ============================================================
# CELL 14 (FINAL): Query the database — test the pipeline
# ============================================================
result = rag_query(
    query        = "Will Malaysian government increase taxes in 2026?",
    child_chunks = child_chunks,
    parent_store = parent_store,
    bm25_index   = bm25_index,
    faiss_index  = faiss_index,
    verbose      = True
)

In [ ]:
# ============================================================
# just change the query and run
# ============================================================
query = "What is the The number of employed persons in Malaysia in March 2026"

result = rag_query(
    query=query,
    child_chunks=child_chunks,
    parent_store=parent_store,
    bm25_index=bm25_index,
    faiss_index=faiss_index,
    verbose=True
)